
# Part 6 — Overfitting and L2 Regularization


In [1]:
import numpy as np
import pandas as pd
import math


## 1. Small Logistic Regression Setup

We use a tiny binary classification dataset so the regularization mechanics stay easy to inspect.


In [2]:
X = np.array([
    [0.5],
    [1.0],
    [1.5],
    [2.0],
    [2.5],
    [3.0],
    [3.5],
    [4.0]
])

y = np.array([0, 0, 0, 0, 1, 1, 1, 1])

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (8, 1)
y shape: (8,)



## 2. Sigmoid, Cost and Gradient

These are the same loop-based implementations I used before regularization.


In [3]:
def sigmoid(z):
    g = 1 / (1 + np.exp(-1 * z))
    return g

In [4]:

def compute_cost(X, y, w, b):
    m, n = X.shape

    total_cost = 0

    for i in range(m):
        total_cost += (-1 * y[i] * np.log(sigmoid(np.dot(X[i], w) + b))) \
                    + (-1 * (1 - y[i]) * np.log(1 - sigmoid(np.dot(X[i], w) + b)))

    total_cost /= m

    return total_cost

In [5]:

def compute_gradient(X, y, w, b):
    m, n = X.shape

    dj_dw = np.zeros(w.shape)
    dj_db = 0.

    for i in range(m):
        f_wb = sigmoid(np.dot(X[i], w) + b)

        dj_db_i = f_wb - y[i]
        dj_db += dj_db_i

        for j in range(n):
            dj_dw_ij = (f_wb - y[i]) * X[i][j]
            dj_dw[j] += dj_dw_ij

    dj_dw /= m
    dj_db /= m

    return dj_db, dj_dw



## 3. Regularized Logistic Cost

Ordinary logistic regression minimizes binary cross-entropy.

L2 regularization adds:

$$
\frac{\lambda}{2m}\sum_{j=1}^{n} w_j^2
$$

So the new cost is:

$$
J_{reg}(w,b)
=
J(w,b)
+
\frac{\lambda}{2m}\sum_{j=1}^{n} w_j^2
$$

The bias term $b$ is not regularized.


In [8]:

def compute_cost_reg(X, y, w, b, lambda_=1):
    m, n = X.shape

    cost_without_reg = compute_cost(X, y, w, b)

    reg_cost = 0.

    for i in range(n):
        reg_cost += (w[i] * w[i])

    reg_cost *= lambda_
    reg_cost /= (2 * m)

    total_cost = cost_without_reg + reg_cost

    return total_cost



## 4. Regularized Gradient

The original weight gradient is:

$$
\frac{\partial J}{\partial w_j}
=
\frac{1}{m}
\sum_{i=1}^{m}
(f^{(i)} - y^{(i)})x_j^{(i)}
$$

L2 regularization adds:

$$
\frac{\lambda}{m}w_j
$$

Therefore:

$$
\frac{\partial J_{reg}}{\partial w_j}
=
\frac{\partial J}{\partial w_j}
+
\frac{\lambda}{m}w_j
$$

The gradient for $b$ stays unchanged.


In [9]:

def compute_gradient_reg(X, y, w, b, lambda_=1):
    m, n = X.shape

    dj_db, dj_dw = compute_gradient(X, y, w, b)

    for i in range(n):
        dj_dw[i] += (lambda_ * w[i]) / m

    return dj_db, dj_dw



## 5. Gradient Descent with Regularization

Gradient descent itself does not change.

The only difference is that we pass the regularized cost and gradient functions.


In [10]:

def gradient_descent(X, y, w_in, b_in, cost_function, gradient_function,
                     alpha, num_iters, lambda_):

    J_history = []

    for i in range(num_iters):

        dj_db, dj_dw = gradient_function(X, y, w_in, b_in, lambda_)

        w_in = w_in - alpha * dj_dw
        b_in = b_in - alpha * dj_db

        if i < 100000:
            cost = cost_function(X, y, w_in, b_in, lambda_)
            J_history.append(cost)

    return w_in, b_in, J_history



## 6. Effect of Lambda

Useful intuition:

- $\lambda = 0$ → no regularization
- small $\lambda$ → more model freedom
- larger $\lambda$ → large weights become more expensive
- extremely large $\lambda$ → weights are pushed toward zero and the model may underfit

Important: $\lambda=0$ does **not** automatically mean overfitting.


In [11]:

lambda_values = [0, 0.1, 1, 10, 100]

results = []

for lambda_ in lambda_values:

    initial_w = np.zeros(X.shape[1])
    initial_b = 0.

    w, b, J_history = gradient_descent(
        X,
        y,
        initial_w,
        initial_b,
        compute_cost_reg,
        compute_gradient_reg,
        alpha=0.1,
        num_iters=5000,
        lambda_=lambda_
    )

    predictions = np.zeros(len(y))

    for i in range(len(y)):
        f_wb = sigmoid(np.dot(X[i], w) + b)

        if f_wb >= 0.5:
            predictions[i] = 1
        else:
            predictions[i] = 0

    accuracy = np.mean(predictions == y)

    results.append({
        "lambda": lambda_,
        "final_regularized_cost": J_history[-1],
        "w": w[0],
        "b": b,
        "|w|": abs(w[0]),
        "train_accuracy": accuracy
    })

pd.DataFrame(results)


,lambda,final_regularized_cost,w,b,|w|,train_accuracy
0,0.0,0.076070,4.691892,-10.407018,4.691892,1.0
1,0.1,0.184020,3.174787,-7.128304,3.174787,1.0
2,1.0,0.390887,1.326579,-2.984802,1.326579,1.0
3,10.0,0.613782,0.318087,-0.715696,0.318087,1.0
4,100.0,0.683403,0.038977,-0.087699,0.038977,1.0



The key thing to observe is the weight magnitude.

As $\lambda$ increases, large weights become more costly, so the learned weights tend to shrink.

That does not mean the training accuracy must immediately fall.  
Regularization is mainly about controlling model flexibility and improving generalization.



## 7. Weight Decay Intuition

The regularized update is:

$$
w_j
\leftarrow
w_j
-
\alpha
\left(
\text{normal gradient}
+
\frac{\lambda}{m}w_j
\right)
$$

Rearranging:

$$
w_j
\leftarrow
\left(1-\frac{\alpha\lambda}{m}\right)w_j
-
\alpha(\text{normal gradient})
$$

So each update slightly shrinks the existing weight before applying the ordinary gradient step.

This is the weight decay intuition behind L2 regularization.



## 8. Takeaways

- Overfitting is associated with high variance.
- Underfitting is associated with high bias.
- L2 regularization adds a penalty for large weights.
- Regularized cost adds `lambda / (2m) * sum(w_j^2)`.
- Regularized gradient adds `lambda / m * w_j`.
- The bias gradient is unchanged.
- Increasing lambda usually reduces model freedom.
- Too much regularization can cause underfitting.
